# 환경설정

In [1]:
# KoNLPy는 내부적으로 Java를 사용합니다
# Java를 먼저 설치해야 KoNLPy가 작동합니다
!apt-get install default-jdk -y

# konlpy : 한국어 형태소 분석 라이브러리
# nltk   : 영어 자연어 처리 라이브러리
# --quiet : 설치 메시지를 간략하게 출력
!pip install konlpy nltk --quiet

# 설치 완료 메시지
print("✅ 설치 완료!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jdk default-jdk-headless default-jre
  default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jn

In [3]:
# 필요한 도구 설치
!pip install gensim --quiet
print("✅ 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 19.0 MB/s eta 0:00:00
✅ 완료!


In [4]:
# 필요한 도구 불러오기
from gensim.models import Word2Vec
import numpy as np

print("✅ 준비 완료!")

✅ 준비 완료!


In [5]:
# ── 라이브러리 불러오기 ──────────────────────

import re                    # 정규표현식 : 텍스트에서 특정 패턴을 찾거나 바꿀 때 사용
from collections import Counter  # Counter : 단어 빈도를 쉽게 셀 때 사용

import nltk                             # 영어 자연어 처리 도구 모음
nltk.download('punkt',     quiet=True)  # 문장/단어 토큰화 모델
nltk.download('punkt_tab', quiet=True)  # 토큰화 추가 데이터
nltk.download('stopwords', quiet=True)  # 영어 불용어 목록
nltk.download('wordnet',   quiet=True)  # 영어 단어 원형 데이터

from konlpy.tag import Okt   # Okt : KoNLPy에서 가장 많이 쓰이는 한국어 형태소 분석기
okt = Okt()                  # 형태소 분석기 객체 생성 (한 번만 만들면 계속 사용 가능)

print("✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀")

✅ 준비 완료! 이제 CH 1 부터 시작하세요 🚀


# 텍스트 전처리

In [6]:
# ── 전처리 파이프라인 함수 정의 ─────────────────
# 모든 단계를 하나로 묶은 함수입니다

# 분석에 불필요한 한국어 단어 목록
불용어_목록 = [
    "것", "수", "나", "저", "제", "그", "이", "때",
    "등", "좀", "잘", "더", "한", "안", "못", "또"
]

def 전처리_파이프라인(텍스트):
    """
    입력: 원본 텍스트 (문자열)
    출력: 전처리된 핵심 단어 리스트
    """
    # 1단계: 정제 — 노이즈 제거
    텍스트 = re.sub(r'http\S+', '', 텍스트)               # URL 제거
    텍스트 = re.sub(r'<[^>]+>', '', 텍스트)               # HTML 태그 제거
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', '', 텍스트)    # 특수문자 제거
    텍스트 = re.sub(r'\s+', ' ', 텍스트).strip()           # 공백 정리

    # 2단계: 정규화 — 반복 문자 압축
    텍스트 = re.sub(r'(.)\1{2,}', r'\1\1', 텍스트)

    # 3단계: 형태소 분석 — 명사 추출
    토큰들 = okt.nouns(텍스트)

    # 4단계: 불용어 제거 + 2글자 이상만 유지
    결과 = [
        단어 for 단어 in 토큰들
        if 단어 not in 불용어_목록  # 불용어 목록에 없는 것만
        and len(단어) >= 2          # 2글자 이상인 것만
    ]

    return 결과

print("✅ 파이프라인 함수 준비 완료!")

✅ 파이프라인 함수 준비 완료!


In [10]:
# ── 과제용 영화 리뷰 데이터 ──────────────────

영화리뷰1 = "아 ㅋㅋ 진짜 미치겠다 세조 싸움 잘함?"
영화리뷰2 = "홍위도 울고 흥도도 울고 나도 울었다"
영화리뷰3 = "오랜만에 사극 영화 보고 웃다가 울다가 과몰입한 영화!! 연기까지 완벽!! 설연휴 영화로 강추"
영화리뷰4 = "단종을 나약하게만 그린 것이 아니라 왕이었다는 것을 느끼게해준 영화. 유해진 박지훈의 조합이 이렇게나 감동을 주다니 여운이 깊게 남습니다."
영화리뷰5 = "와 오랜만에 극장에서 볼만한 영화 나온듯배우들 연기 차력쇼 미쳤고, 보면서 눈물 콧물 다 흘림 진짜로.. 설 연휴에 부모님이랑 같이 보기 좋은 듯!!"
영화리뷰6 = "가족들과 한번 더 볼거에요.... 유해진 박지훈 연기 미쳤고, 웃고 우느라 속절없이 당해버림. 단종은 박지훈.."
영화리뷰7 = "오ㅏ.. 결과를 알고봐도 좋았고 설에 부모님 모시고 또 봐야지"
영화리뷰8 = "박지훈 눈을 봐라 단종이다"
영화리뷰9 = "박지훈 퍼스널컬러는 단종이였다"
영화리뷰10 = "버석한 입술에 미소가 걸리고 텅 빈 눈에 다시 빛이 돌아올 때, 소년은 그제야 어른이 된다."
영화리뷰11 = "박지훈은 영화계 최고의 보석이다. 눈에 보석박음.. 눈만봐도 눈물.."
영화리뷰12 = "박지훈 연기…미쳤구낭"
영화리뷰13 = "박지훈의 단종이 너무너무너무 인상깊고 연기가 미쳤습니다 20대 배우중 제일 기대되는 배우그리고 유해진이 날아다닙니다 왕사남으로 큰상 많이 받으실듯"
영화리뷰14 = "마지막에 오열해버렸습니다"
영화리뷰15 = "단종은 하고 싶어서 하는 역할이 아니라 주어진 자가 할 수 있는 역할이라는데 그 주어진 자가 박지훈인게 확실하다. 향후 10년동안은 단종하면 박지훈을 떠올릴듯"
영화리뷰16 = "유해진 박지훈 두 사람 상줘라"
영화리뷰17 = "이런 사극 오랜만인데 너무 좋았습니다 ㅠㅠ 박지훈 대배우될 거 같음"
영화리뷰18 = "오늘 이후로 김은희 남편이 아닌 <왕과 사는 남자>의 감독으로 불릴것! 결과를 우리 모두 다 알고 있기에 영화에서의 결말만큼이라도 성공하기를 바랬다 ㅠㅠ"
영화리뷰19 = "그냥.. 보세요.. 진짜 이런 감정을 느끼게 해준 영화가 얼마만인지... 감히 올해의 영화라 할 수 있다고 봅니다"
영화리뷰20 = "유해진 드디어 남우주연상에 오를듯,,,,마지막 클라이막스는 유해진이 아니면 살릴수없는 연기"
영화리뷰21 = "진짜 미쳤다 웃다가 울다가 마지막엔 꺼이꺼이 울었음 천만 넘을거 같다"
영화리뷰22 = "유해진 유지태야 뭐 원래 잘하는거 알고있었는데 박지훈 연기 진짜 미쳤네.."
영화리뷰23 = "박지훈은 눈 하나만으로도 일 끊길 일은 없겠다 싶다"
영화리뷰24 = "연기구멍은 호랑이 뿐, 배우들 연기가 미쳤습니다!! 그냥 엄흥도랑 단종 데려다놓은줄"
영화리뷰25 = "재미도 있지만, 엄마랑 둘이 광광 울었네요..박지훈님 그 눈망울이 계속 머릿속에 맴돌아요"
영화리뷰26 = "오랜만에 극장에서 본 사극이었는데 역사 찢고 나온 배우들의 열연이 정말 최고! 휴지 꼭 잊지 말고 챙겨가길!"
영화리뷰27 = "어렸을때부터 왕으로 자란 단종의 느낌이 자연스럽게 표현되어 진짜 단종이라면 저랬을것 같았어요그리고 박지훈배우의 얼굴이한몫을 합니다유해진배우님이 말씀하신 마지막 밤씬은정말최고였습니다연출적으로 아쉬운부분은 있지만저는 또보러갈겁니다"
영화리뷰28 = "유지태 표독 눈매 리프팅 미쳤다"
영화리뷰29 = "호감 배우들의 연기파티에 실화의 묵직함까지, 항주니 감독님의 최고의 영화가 될듯.."

print("영화 리뷰 5개 준비 완료!")
print()
print("1:", 영화리뷰1)
print("2:", 영화리뷰2)
print("3:", 영화리뷰3)
print("4:", 영화리뷰4)
print("5:", 영화리뷰5)

영화 리뷰 5개 준비 완료!

1: 아 ㅋㅋ 진짜 미치겠다 세조 싸움 잘함?
2: 홍위도 울고 흥도도 울고 나도 울었다
3: 오랜만에 사극 영화 보고 웃다가 울다가 과몰입한 영화!! 연기까지 완벽!! 설연휴 영화로 강추
4: 단종을 나약하게만 그린 것이 아니라 왕이었다는 것을 느끼게해준 영화. 유해진 박지훈의 조합이 이렇게나 감동을 주다니 여운이 깊게 남습니다.
5: 와 오랜만에 극장에서 볼만한 영화 나온듯배우들 연기 차력쇼 미쳤고, 보면서 눈물 콧물 다 흘림 진짜로.. 설 연휴에 부모님이랑 같이 보기 좋은 듯!!


In [14]:
영화_문장들 = []  # 빈 리스트 준비
for i in range(0,29):
  영화_문장들.append(전처리_파이프라인(eval(f"영화리뷰{i+1}")))

print("전체 단어 목록:", 영화_문장들)

전체 단어 목록: [['진짜', '세조', '싸움'], ['위도', '도도'], ['사극', '영화', '보고', '다가', '몰입', '영화', '연기', '완벽', '설연휴', '영화로', '강추'], ['단종', '그린', '영화', '유해진', '박지훈', '조합', '감동', '여운'], ['극장', '영화', '배우', '연기', '차력', '눈물', '콧물', '진짜', '연휴', '부모님', '보기'], ['가족', '한번', '유해진', '박지훈', '연기', '단종', '박지훈'], ['결과', '부모님', '모시'], ['박지훈', '단종'], ['박지훈', '퍼스', '컬러', '종이'], ['입술', '미소', '다시', '소년', '어른'], ['박지훈', '영화계', '최고', '보석', '보석', '눈물'], ['박지훈', '연기'], ['박지훈', '단종', '인상', '연기', '배우', '제일', '기대', '배우', '유해진', '왕사'], ['마지막', '오열'], ['단종', '역할', '자가', '역할', '자가', '박지훈', '향후', '단종', '박지훈'], ['유해진', '박지훈', '사람'], ['사극', '박지훈', '배우'], ['오늘', '이후', '김은희', '남편', '감독', '결과', '우리', '모두', '영화', '결말', '바랬다'], ['그냥', '진짜', '감정', '영화', '얼마', '감히', '올해', '영화'], ['유해진', '남우', '주연', '마지막', '막스', '유해진', '연기'], ['진짜', '다가', '마지막', '천만'], ['유해진', '유지태', '원래', '박지훈', '연기', '진짜'], ['박지훈', '하나'], ['구멍', '호랑이', '배우', '연기', '그냥', '엄흥도', '단종', '은줄'], ['재미', '엄마', '박지훈', '눈망울', '계속', '머릿속'], ['극장', '사극', '역사', '배우', '열연', '정말', '최고', '

# TF-IDF

In [15]:
# ── Word2Vec 학습 (핵심 3줄) ────────────────
model = Word2Vec(
    sentences=영화_문장들,   # 입력: 단어 리스트의 리스트
    vector_size=30,     # 각 단어를 30차원 벡터로 표현
    window=4,           # 앞뒤 4단어를 문맥으로 봄
    min_count=1,        # 1번 이상 등장한 단어 포함
    epochs=500,         # 500번 반복 학습
    seed=42
)

print("학습 완료!")
print(f"어휘 크기: {len(model.wv)}개 단어")
print(f"벡터 차원: {model.vector_size}차원")

학습 완료!
어휘 크기: 106개 단어
벡터 차원: 30차원


In [17]:
# ── 결과 확인 1: 유사한 단어 찾기 ─────────
print("'연기'와 비슷한 단어:")
for 단어, 유사도 in model.wv.most_similar("연기", topn=5):
    막대 = "█" * int(유사도 * 20)
    print(f"  '{단어}': {유사도:.4f}  {막대}")

print()
print("'감동'과 비슷한 단어:")
for 단어, 유사도 in model.wv.most_similar("감동", topn=5):
    막대 = "█" * int(유사도 * 20)
    print(f"  '{단어}': {유사도:.4f}  {막대}")

print()
print("'박지훈'과 비슷한 단어:")
for 단어, 유사도 in model.wv.most_similar("박지훈", topn=5):
    막대 = "█" * int(유사도 * 20)
    print(f"  '{단어}': {유사도:.4f}  {막대}")

'연기'와 비슷한 단어:
  '진짜': 0.9453  ██████████████████
  '부모님': 0.9428  ██████████████████
  '원래': 0.9405  ██████████████████
  '왕사': 0.9377  ██████████████████
  '영화로': 0.9364  ██████████████████

'감동'과 비슷한 단어:
  '그린': 0.9862  ███████████████████
  '조합': 0.9833  ███████████████████
  '여운': 0.9792  ███████████████████
  '세조': 0.9328  ██████████████████
  '원래': 0.9315  ██████████████████

'박지훈'과 비슷한 단어:
  '단종': 0.9693  ███████████████████
  '한번': 0.9295  ██████████████████
  '향후': 0.9143  ██████████████████
  '유해진': 0.9102  ██████████████████
  '가족': 0.9020  ██████████████████


In [20]:
# ── 결과 확인 2: 두 단어의 유사도 ─────────
쌍들 = [
    ("연기", "배우"),        # 같이 등장 → 높아야 함
    ("박지훈", "단종"),        # 같이 등장 → 높아야 함
    ("박지훈", "연기"),    # 같이 등장 → 높아야 함
    ("미소", "오열"),    # 다른 문맥 → 낮아야 함
    ("단종", "엄흥도"),          # 같이 등장 → 높아야 함
]

print("단어 쌍 유사도 (1.0 = 완전 동일, 0.0 = 완전 다름):")
print("-" * 46)
for 단어1, 단어2 in 쌍들:
    유사도 = model.wv.similarity(단어1, 단어2)
    막대 = "█" * int(유사도 * 20)
    print(f"  '{단어1}' ↔ '{단어2}': {유사도:.4f}  {막대}")

단어 쌍 유사도 (1.0 = 완전 동일, 0.0 = 완전 다름):
----------------------------------------------
  '연기' ↔ '배우': 0.8734  █████████████████
  '박지훈' ↔ '단종': 0.9693  ███████████████████
  '박지훈' ↔ '연기': 0.8690  █████████████████
  '미소' ↔ '오열': 0.8612  █████████████████
  '단종' ↔ '엄흥도': 0.8678  █████████████████


In [21]:
# ── 결과 확인 3: 단어 벡터 출력 ────────────
print("'연기'의 벡터 (30차원):")
print(model.wv["연기"].round(3))
print()
print("'배우'의 벡터 (30차원):")
print(model.wv["배우"].round(3))
print()
print("두 벡터의 숫자 패턴이 비슷하면 의미가 유사한 것입니다.")

'연기'의 벡터 (30차원):
[ 0.462 -0.09  -0.184 -0.019  0.03  -0.37   0.247  0.283  0.108 -0.199
 -0.253 -0.118  0.191 -0.072 -0.037  0.436  0.068 -0.398 -0.152  0.291
  0.441 -0.058 -0.411  0.283  0.936  0.064 -0.16  -0.237 -0.417 -0.009]

'배우'의 벡터 (30차원):
[ 0.476 -0.071 -0.248  0.421  0.144 -0.424  0.225  0.236  0.01  -0.445
 -0.288 -0.194  0.248 -0.214 -0.325  0.53   0.334 -0.158 -0.19   0.563
  0.403  0.006 -0.362  0.353  0.821 -0.338 -0.188 -0.534 -0.382 -0.023]

두 벡터의 숫자 패턴이 비슷하면 의미가 유사한 것입니다.


# Word2Vec

In [23]:
# 이 아이디어를 우리 데이터에서 직접 확인합니다
print("우리 학습 데이터에서 '연기'와 '배우'의 문맥:")
print()
for i, 문장 in enumerate(영화_문장들):
    if "연기" in 문장 or "배우" in 문장:
        print(f"  문장{i+1:2}: {문장}")

우리 학습 데이터에서 '연기'와 '배우'의 문맥:

  문장 3: ['사극', '영화', '보고', '다가', '몰입', '영화', '연기', '완벽', '설연휴', '영화로', '강추']
  문장 5: ['극장', '영화', '배우', '연기', '차력', '눈물', '콧물', '진짜', '연휴', '부모님', '보기']
  문장 6: ['가족', '한번', '유해진', '박지훈', '연기', '단종', '박지훈']
  문장12: ['박지훈', '연기']
  문장13: ['박지훈', '단종', '인상', '연기', '배우', '제일', '기대', '배우', '유해진', '왕사']
  문장17: ['사극', '박지훈', '배우']
  문장20: ['유해진', '남우', '주연', '마지막', '막스', '유해진', '연기']
  문장22: ['유해진', '유지태', '원래', '박지훈', '연기', '진짜']
  문장24: ['구멍', '호랑이', '배우', '연기', '그냥', '엄흥도', '단종', '은줄']
  문장26: ['극장', '사극', '역사', '배우', '열연', '정말', '최고', '휴지']
  문장27: ['단종', '느낌', '자연', '표현', '진짜', '종이', '라면', '박지훈', '배우', '얼굴', '한몫', '유해진', '배우', '말씀', '마지막', '밤씬', '정말', '최고', '연출', '부분']
  문장29: ['호감', '배우', '파티', '실화', '항주', '감독', '최고', '영화']


In [25]:
# 긍정 단어끼리, 부정 단어끼리 비교
print("긍정 관련 단어 유사도:")
print(f"  감동 ↔ 여운:      {model.wv.similarity('감동','여운'):.4f}")
print(f"  연기 ↔ 배우:      {model.wv.similarity('연기','배우'):.4f}")
print()

긍정 관련 단어 유사도:
  감동 ↔ 여운:      0.9792
  연기 ↔ 배우:      0.8734



# Word2Vec 파라미터

In [26]:
# ── window: 문맥 범위 ──────────────────────
# 예시 문장: ["배우", "표정", "눈빛", "감정", "전달"]
# window=2일 때 '눈빛' 기준: [표정, 감정] 만 봄
# window=4일 때 '눈빛' 기준: [배우, 표정, 감정, 전달] 다 봄

model_w2 = Word2Vec(sentences=영화_문장들, vector_size=30, window=2, min_count=1, epochs=500, seed=42)
model_w6 = Word2Vec(sentences=영화_문장들, vector_size=30, window=6, min_count=1, epochs=500, seed=42)

print("[window 비교] — 연기↔배우 유사도")
print(f"  window=2: {model_w2.wv.similarity('연기','배우'):.4f}")
print(f"  window=4: {model.wv.similarity('연기','배우'):.4f}  ← 현재 설정")
print(f"  window=6: {model_w6.wv.similarity('연기','배우'):.4f}")
print()
print("💡 실전: 보통 5 사용 / 문장이 짧으면 3~4")

[window 비교] — 연기↔배우 유사도
  window=2: 0.9363
  window=4: 0.8734  ← 현재 설정
  window=6: 0.8243

💡 실전: 보통 5 사용 / 문장이 짧으면 3~4


In [27]:
# ── epochs: 반복 학습 횟수 ──────────────────
# 적게: 학습 부족 (덜 수렴)
# 많이: 더 잘 배움 (시간은 더 걸림)

model_e20  = Word2Vec(sentences=영화_문장들, vector_size=30, window=4, min_count=1, epochs=20,  seed=42)
model_e100 = Word2Vec(sentences=영화_문장들, vector_size=30, window=4, min_count=1, epochs=100, seed=42)

print("[epochs 비교] — 연기↔배우 유사도")
print(f"  20번:  {model_e20.wv.similarity('연기','배우'):.4f}")
print(f"  100번: {model_e100.wv.similarity('연기','배우'):.4f}")
print(f"  500번: {model.wv.similarity('연기','배우'):.4f}  ← 현재 설정")
print()
print("💡 실전: 데이터 적으면 epochs 늘리기 / 대용량 데이터는 10~30으로도 충분")

[epochs 비교] — 연기↔배우 유사도
  20번:  -0.1622
  100번: 0.8597
  500번: 0.8734  ← 현재 설정

💡 실전: 데이터 적으면 epochs 늘리기 / 대용량 데이터는 10~30으로도 충분


# 유사 단어 검색

In [34]:
# most_similar() 동작 원리
# 1. 입력 단어의 벡터를 가져옴
# 2. 모든 단어 벡터와 코사인 유사도 계산
# 3. 높은 순으로 정렬 → 상위 N개 반환

검색_단어들 = ["단종", "눈물", "가족", "연기"]

for 기준 in 검색_단어들:
    유사어 = model.wv.most_similar(기준, topn=4)
    top = [f"'{w}'({s:.2f})" for w,s in 유사어]
    print(f"  '{기준}' 유사어: {', '.join(top)}")

  '단종' 유사어: '박지훈'(0.97), '향후'(0.92), '역할'(0.91), '자가'(0.90)
  '눈물' 유사어: '보기'(0.99), '콧물'(0.98), '연휴'(0.98), '차력'(0.98)
  '가족' 유사어: '사람'(0.96), '한번'(0.95), '싸움'(0.94), '세조'(0.94)
  '연기' 유사어: '진짜'(0.95), '부모님'(0.94), '원래'(0.94), '왕사'(0.94)


In [35]:
model.wv["연기"]

array([ 0.4620154 , -0.09024137, -0.18373191, -0.01883322,  0.02974391,
       -0.37041578,  0.24738137,  0.28349355,  0.10790453, -0.19853854,
       -0.25335672, -0.11844962,  0.19074786, -0.07158732, -0.03671119,
        0.43595228,  0.06823806, -0.39788803, -0.1515558 ,  0.29118973,
        0.44090533, -0.05799717, -0.41087565,  0.2828695 ,  0.9356112 ,
        0.06394043, -0.15988085, -0.2370881 , -0.41679633, -0.00901456],
      dtype=float32)

In [36]:
model.wv.most_similar("연기", topn=5)   # 유사 단어

[('진짜', 0.9452947974205017),
 ('부모님', 0.9427801370620728),
 ('원래', 0.940450131893158),
 ('왕사', 0.937692403793335),
 ('영화로', 0.9364498257637024)]

In [37]:
model.wv.similarity("연기", "배우")    # 유사도

np.float32(0.87342775)

In [38]:
"연기" in model.wv                       # 어휘 확인

True

In [ ]:
model.save("model.model")               # 저장